# S6E10 — 09 Richer features for the neural network (`X_rich`)

> **Pipeline position:** 03 `X_final` → 07 RealMLP → 08 TabPFN → **09 rich features** → 10 RealMLP-rich → 11 stack

| | |
|---|---|
| **Input** | `data/raw/*` (incl. `orig_data.csv`), `data/processed/X_final_*.parquet` |
| **Output** | `data/processed/X_rich_train.parquet`, `X_rich_test.parquet` — `X_final` (45 columns) + 47 new = **92 columns** |
| **Runtime** | ~2 min (CatBoost captain ≈ 1 min); `pip install tiktoken` once |
| **Labels used?** | **No competition labels** (so no fold logic). Target-encoded keys are built per fold in notebook 09. |

**Goal:** `07` showed that RealMLP on `X_final` reaches 0.96082 while a public notebook reports 0.96123 for the same recipe. Their difference is mainly **richer input features**. This notebook builds those families so `10` can test whether they help *our* pipeline.

| # | Family | Columns | Idea |
|---|---|---|---|
| 1 | Digits | 7 (`fd_mod10`, `fd_mod100`, `fd_mod1000`, `fd_div10`, `fd_div100`, `fd_length`, `age_mod10`) | The generator leaves traces in the last digits of distance and age |
| 2 | GPT-2 token keys | 12 (first / last token / count for distance, age, both delays) | A number written as text splits into sub-word tokens; their ids are extra fingerprints of a value |
| 3 | Delays | 2 (`total_delay`, `delay_recovered`) | Sum and difference of departure/arrival delay |
| 4 | Original-survey rates | 22 (`og_rate_*`: 21 raw columns + the travel/class/customer trio) | Satisfaction rate in the *original* survey for each value |
| 5 | Second captain | 1 (`og_cat_logit`) | CatBoost trained only on the original survey, scoring competition rows |
| 6 | Keys for fold-wise TE | 3 (`key_age_class`, `key_fd_class`, `key_age_travel`) | Combined ids, target-encoded **inside each fold** in `10` |

> **Glossary:** *Captain* = a model trained on the original survey whose prediction is used as a feature for the competition data. *Byte-pair encoding (BPE)* = how GPT-2 splits text into sub-word tokens.

In [1]:
import re, time, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings('ignore')
SMOKE = False      # True: CatBoost captain with 100 trees (a few seconds)

RAW = next(p for p in [Path('data/raw'), Path('../data/raw'), Path('/kaggle/input/playground-series-s6e10')] if (p / 'train.csv').exists())
PROC = next(p for p in [Path('data/processed'), Path('../data/processed'), Path('/kaggle/working'), Path('.')] if (p / 'X_final_train.parquet').exists())
ORIG = next(p for p in [RAW / 'orig_data.csv', Path('/kaggle/input/s6e10-orig-data/orig_data.csv')] if p.exists())

train = pd.read_csv(RAW / 'train.csv'); test = pd.read_csv(RAW / 'test.csv'); orig = pd.read_csv(ORIG)
Xf_tr = pd.read_parquet(PROC / 'X_final_train.parquet'); Xf_te = pd.read_parquet(PROC / 'X_final_test.parquet')
assert len(train) == len(Xf_tr) == 699635 and len(test) == len(Xf_te) == 299844
assert (train['Age'].values == Xf_tr['age'].values).all() and (train['Flight Distance'].values == Xf_tr['flight_distance'].values).all()
assert (test['Age'].values == Xf_te['age'].values).all() and (test['Flight Distance'].values == Xf_te['flight_distance'].values).all()
print('rows aligned with X_final | original survey rows:', len(orig))

# The satisfaction column in the original survey is text (satisfied / neutral or dissatisfied); convert to 0/1.
def tobin(s):
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0', 'satisfied']).astype(int)
y_orig = tobin(orig['satisfaction'])
CATS = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
NUMS = ['Age', 'Flight Distance', 'Departure Delay in Minutes', 'Arrival Delay in Minutes']
RATINGS = [c for c in test.columns if c not in CATS + NUMS + ['id']]
BASE = NUMS + RATINGS + CATS
assert len(RATINGS) == 13
clean = lambda s: re.sub(r'[^0-9a-zA-Z]+', '_', s).strip('_').lower()
full = pd.concat([train[BASE], test[BASE]], ignore_index=True)
n_train = len(train)
# All new columns collect in E; they are appended to X_final at the end. Train and test are processed together (no labels involved), index = train rows first, then test rows.
E = pd.DataFrame(index=full.index)          # the new columns

rows aligned with X_final | original survey rows: 129880


## 1. Digits and delays
Cheap arithmetic features on distance, age and delays. `delay_recovered = departure − arrival` shows whether the plane made up time in the air.

In [2]:
# Digit features: if the data generator produced distances/ages with a trace in the last digits, trees and nets can pick it up.
fd, age = full['Flight Distance'], full['Age']
E['fd_mod10'], E['fd_mod100'], E['fd_mod1000'] = fd % 10, fd % 100, fd % 1000
E['fd_div10'], E['fd_div100'], E['age_mod10'] = fd // 10, fd // 100, age % 10
E['fd_length'] = fd.astype(int).astype(str).str.len()
dep, arr = full['Departure Delay in Minutes'], full['Arrival Delay in Minutes']
# Fill missing arrival delay with departure delay (same rule as in 03).
arr_f = arr.fillna(dep)
E['total_delay'] = dep + arr_f
E['delay_recovered'] = dep - arr_f
print('digits + delays:', E.shape[1], 'columns')

digits + delays: 9 columns


## 2. GPT-2 token keys
Requires `tiktoken` (`pip install tiktoken`; the first run downloads the vocabulary — without internet the block is skipped with a warning, and `10` still runs, with fewer columns). This is an unusual trick taken from the public notebook. `10` measured it: RealMLP on fold 0 with the tokens (92 columns) scored 0.96134, without them (80 columns) 0.96133 — **no effect**. The columns stay in `X_rich` only for reproducibility (the run without them is saved as `oof_realmlp_rich80.npy`).

In [3]:
try:
    import tiktoken
    # GPT-2 byte-pair encoding turns a number into 1-2 sub-word tokens. The first/last token and token count act as extra "fingerprints" of a value; we simply test whether they carry signal.
    GPT2 = tiktoken.get_encoding('gpt2')
    for col, nm in [('Flight Distance', 'fd'), ('Age', 'age'), ('Departure Delay in Minutes', 'dep'), ('Arrival Delay in Minutes', 'arr')]:
        # One encoding per UNIQUE value (fast), then mapped to all rows.
        toks = {v: GPT2.encode(' ' + str(int(v))) for v in full[col].dropna().unique()}
        E[f'{nm}_tok_first'] = full[col].map({v: t[0] for v, t in toks.items()})
        E[f'{nm}_tok_last'] = full[col].map({v: t[-1] + 100_000 * len(t) for v, t in toks.items()})
        E[f'{nm}_tok_count'] = full[col].map({v: len(t) for v, t in toks.items()})
    print('token keys added; example distance 3983 ->', GPT2.encode(' 3983'))
except Exception as err:
    print('WARNING: GPT-2 token keys skipped:', str(err)[:150])

token keys added; example distance 3983 -> [39260, 18]


## 3. Original-survey rates (no competition labels)
For each raw column and each value: share of satisfied passengers among **original-survey** passengers with that value. These are target-encoding-like features, but the label source is the original dataset, so there is no leakage into the competition folds. 100% of competition distances occur in the original data, so the lookup rarely falls back to the global rate.

In [4]:
# Fallback: a value absent from the original survey gets the overall original satisfaction rate.
glob_rate = y_orig.mean()
# For each raw column: share of satisfied passengers in the ORIGINAL survey among those with the same value. Uses original labels only, never competition labels.
for col in BASE:
    rate = y_orig.groupby(orig[col].values).mean()
    E[f'og_rate_{clean(col)}'] = full[col].map(rate).fillna(glob_rate)
trio = ['Type of Travel', 'Class', 'Customer Type']
trio_rate = orig[trio].assign(y=y_orig.values).groupby(trio)['y'].mean().rename('trio_og_rate')
E['og_rate_trio'] = full[trio].merge(trio_rate, left_on=trio, right_index=True, how='left')['trio_og_rate'].fillna(glob_rate).values
print('original-survey rates:', sum(c.startswith('og_rate') for c in E.columns), 'columns | share of distances found in the original:',
      round(full['Flight Distance'].isin(set(orig['Flight Distance'])).mean(), 4))

original-survey rates: 22 columns | share of distances found in the original: 1.0


## 4. Second captain: CatBoost trained only on the original survey
Trained on 129 880 original rows. Its AUC on competition train (labels used **only to report the score**, not for training) is 0.9544 — almost the same as the LightGBM captain's 0.95472 — i.e. the original survey really predicts competition satisfaction, and the two captains give slightly different views.

In [5]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
CAT_MAPS = {c: {v: i for i, v in enumerate(sorted(pd.concat([train[c], test[c]]).dropna().unique()))} for c in CATS}
def enc(df):
    X = df[BASE].copy()
    for c in CATS: X[c] = X[c].map(CAT_MAPS[c]).fillna(-1).astype('int16')
    return X
# Train on complete, de-duplicated original rows only.
og = orig.dropna(subset=[c for c in BASE if c != 'Arrival Delay in Minutes'])[BASE + ['satisfaction']].drop_duplicates()
t0 = time.time()
# Second "captain": CatBoost (handles categories natively) as an alternative view next to the LightGBM captain from 03.
cb = CatBoostClassifier(iterations=100 if SMOKE else 1500, learning_rate=0.05, depth=8, cat_features=CATS, random_seed=42, verbose=0,
                        allow_writing_files=False, thread_count=-1)
cb.fit(enc(og), tobin(og['satisfaction']).values)
p = cb.predict_proba(enc(full))[:, 1].clip(1e-5, 1 - 1e-5)
E['og_cat_logit'] = np.log(p / (1 - p))
y_tr = tobin(train['satisfaction']).values
print(f'captain trained on {len(og):,} original rows in {time.time()-t0:.0f}s | AUC on competition train (no labels used): {roc_auc_score(y_tr, p[:n_train]):.4f}  (the LightGBM captain gives 0.95472)')

captain trained on 129,880 original rows in 62s | AUC on competition train (no labels used): 0.9544  (the LightGBM captain gives 0.95472)


## 5. Keys for fold-wise target encoding, assemble and save
The three `key_*` columns are only identifiers for target encoding in `10` (they are dropped from the network input). The final frame is checked for NaN and saved as parquet.

In [6]:
cls = full['Class'].map(CAT_MAPS['Class']); trv = full['Type of Travel'].map(CAT_MAPS['Type of Travel'])
E['key_age_class'] = age * 10 + cls
E['key_fd_class'] = fd * 10 + cls
E['key_age_travel'] = age * 10 + trv

# NaN appears only in token columns where a delay is missing; -1 marks it.
E = E.fillna(-1).astype('float32')     # NaN only where a delay is missing (token columns); -1 = missing, the flag arrival_delay_missing already exists
Etr, Ete = E.iloc[:n_train].reset_index(drop=True), E.iloc[n_train:].reset_index(drop=True)
Xr_tr = pd.concat([Xf_tr.reset_index(drop=True), Etr], axis=1); Xr_te = pd.concat([Xf_te.reset_index(drop=True), Ete], axis=1)
assert not Xr_tr.isna().any().any() or print('NaN columns:', Xr_tr.columns[Xr_tr.isna().any()].tolist())
Xr_tr.to_parquet(PROC / 'X_rich_train.parquet'); Xr_te.to_parquet(PROC / 'X_rich_test.parquet')
print('saved X_rich:', Xr_tr.shape, Xr_te.shape, '| new columns:', E.shape[1]); print(list(E.columns))

saved X_rich: (699635, 92) (299844, 92) | new columns: 47
['fd_mod10', 'fd_mod100', 'fd_mod1000', 'fd_div10', 'fd_div100', 'age_mod10', 'fd_length', 'total_delay', 'delay_recovered', 'fd_tok_first', 'fd_tok_last', 'fd_tok_count', 'age_tok_first', 'age_tok_last', 'age_tok_count', 'dep_tok_first', 'dep_tok_last', 'dep_tok_count', 'arr_tok_first', 'arr_tok_last', 'arr_tok_count', 'og_rate_age', 'og_rate_flight_distance', 'og_rate_departure_delay_in_minutes', 'og_rate_arrival_delay_in_minutes', 'og_rate_inflight_wifi_service', 'og_rate_departure_arrival_time_convenient', 'og_rate_ease_of_online_booking', 'og_rate_gate_location', 'og_rate_food_and_drink', 'og_rate_online_boarding', 'og_rate_seat_comfort', 'og_rate_inflight_entertainment', 'og_rate_on_board_service', 'og_rate_leg_room_service', 'og_rate_baggage_handling', 'og_rate_checkin_service', 'og_rate_cleanliness', 'og_rate_gender', 'og_rate_customer_type', 'og_rate_type_of_travel', 'og_rate_class', 'og_rate_trio', 'og_cat_logit', 'key

## Summary & decisions
- **Result:** `X_rich` = **92 columns** (45 + 47). 12 token columns were created (example: distance 3983 → tokens `[39260, 18]`).
- **Why not one big notebook with 10?** Features are label-free and deterministic: build once, reuse in any later model. `10` only adds what needs the labels (in-fold TE).
- **Verdict on usefulness is in `10`:** feature families are *hypotheses*; the CV gain of RealMLP on `X_rich` vs `X_final` decides whether they stay. First result: the GPT-2 token block adds nothing (fold 0: 0.96134 with vs 0.96133 without), so the gain comes from the other families.
- **Next:** `10_realmlp_rich.ipynb`.